# EDA pre-modelado — tfm_energia

Punto de partida para el analisis exploratorio antes de estructurar el dataset de modelado predictivo.

Rama: `willy_test` · Fuente: PostgreSQL `tfm_energia` (91.134.143.153) · Credenciales: `ingesta/credentials.json`

## 0. Setup — conexión a la base de datos

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "ingesta"))
from config import load_config

import pandas as pd
import psycopg2

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

_, DB_CONFIG = load_config()
conn = psycopg2.connect(**DB_CONFIG)

print("Conectado a", DB_CONFIG["host"], "/", DB_CONFIG["dbname"])

Conectado a 91.134.143.153 / tfm_energia


## 1. Inventario de tablas — qué hay y qué falta

Recuento rápido de filas y rango de fechas de las 18 tablas, para no arrancar el EDA a ciegas.
Ajustado a lo que ya sabemos (ver el artifact de auditoría): 11 tablas con datos, 6 vacías, 1 legacy.

In [2]:
# tabla: columna de tiempo — actualizado tras la migración del 14-15 agosto
# (esios_marketdata / entsoe_data / entsoe_prices_da ya no existen)
TABLAS = {
    "entsoe_gen_data":            "datetime",
    "entsoe_load_inter":          "datetime",
    "esios_gen":                  "datetime",
    "esios_load_inter":           "datetime",
    "esios_forecast_da":          "datetime",
    "esios_capacity_available":   "date",
    "esios_capacity_installed":   "date",
    "commodities":                "fecha",
    "era5_weather_agg":           "ts",
    "ecmwf_forecast_agg":         "ts",
    "spot_price":                 "datetime",
    "entsoe_forecast_da":         "datetime",
    "esios_pbf_bilateral":        "datetime",
    "esios_pbf_gen":              "datetime",
    "esios_pbf_load_inter":       "datetime",
    "eua_next_dec":               "fecha",
    "ttf_m1":                     "fecha",
    "trayport_daily":             "fecha",
    "trayport_daily_ohlc":        "fecha",
}

resumen = []
with conn.cursor() as cur:
    for tabla, col in TABLAS.items():
        cur.execute(f"SELECT COUNT(*), MIN({col}), MAX({col}) FROM {tabla}")
        n, mn, mx = cur.fetchone()
        resumen.append({"tabla": tabla, "col_tiempo": col, "filas": n, "desde": mn, "hasta": mx})

df_inventario = pd.DataFrame(resumen).sort_values("filas", ascending=False).reset_index(drop=True)
df_inventario

,tabla,col_tiempo,filas,desde,hasta
0,esios_forecast_da,datetime,58150,2019-12-31 01:00:00+01:00,2026-08-18 23:00:00+02:00
1,esios_pbf_load_inter,datetime,58127,2020-01-01 00:00:00+01:00,2026-08-18 23:00:00+02:00
2,esios_pbf_gen,datetime,58127,2020-01-01 00:00:00+01:00,2026-08-18 23:00:00+02:00
3,spot_price,datetime,58121,2020-01-01 00:00:00+01:00,2026-08-18 23:00:00+02:00
4,entsoe_forecast_da,datetime,58097,2020-01-01 00:00:00+01:00,2026-08-17 23:00:00+02:00
5,esios_pbf_bilateral,datetime,58091,2020-01-01 00:00:00+01:00,2026-08-18 23:00:00+02:00
6,esios_load_inter,datetime,58079,2020-01-01 00:00:00+01:00,2026-08-16 23:00:00+02:00
7,esios_gen,datetime,58055,2020-01-01 00:00:00+01:00,2026-08-15 23:00:00+02:00
8,entsoe_gen_data,datetime_utc,58054,2020-01-01 01:00:00+01:00,2026-08-15 23:00:00+02:00
9,entsoe_load_inter,datetime_utc,58054,2020-01-01 01:00:00+01:00,2026-08-15 23:00:00+02:00


## 2. Carga de las tablas núcleo (familia horaria UTC)

`entsoe_gen_data`, `entsoe_load_inter`, `esios_gen`, `esios_load_inter` y `esios_forecast_da`
comparten histórico 2020→hoy y granularidad horaria. `esios_gen`/`esios_load_inter` reemplazan a
`esios_marketdata` desde la migración del 14-ago — ver sección 00 del artifact de auditoría.

⚠️ **Trampa detectada al probar este notebook**: `pd.read_sql(..., parse_dates=[...])` sobre una
columna `timestamptz` de Postgres que cruza un cambio de hora (CET ↔ CEST) la deja en
`datetime64[us, UTC+01:00]` — un offset *fijo* — y convierte en `NaT` **la mitad de las filas**
(las que están en el otro offset). Si luego haces un merge por esa columna, esos `NaT` casan entre
sí y el resultado explota (lo vimos: un JOIN de 31.583 × 31.583 filas intentó reservar 48 GB).
Por eso aquí se lee **sin** `parse_dates` y se normaliza a UTC a mano con
`pd.to_datetime(..., utc=True)` justo después.

In [3]:
# Ventana de trabajo — ampliar cuando el pipeline de EDA esté maduro
START = "2023-01-01"
END   = "2026-08-15"

def leer_horaria(sql: str, col_ts: str) -> pd.DataFrame:
    """SELECT + normalizacion UTC manual (ver aviso arriba: NUNCA parse_dates aqui)."""
    df = pd.read_sql(sql, conn, params={"start": START, "end": END})
    df[col_ts] = pd.to_datetime(df[col_ts], utc=True)
    return df

df_gen = leer_horaria(
    "SELECT * FROM entsoe_gen_data WHERE datetime BETWEEN %(start)s AND %(end)s ORDER BY datetime",
    "datetime",
)
df_load = leer_horaria(
    "SELECT * FROM entsoe_load_inter WHERE datetime BETWEEN %(start)s AND %(end)s ORDER BY datetime",
    "datetime",
)
df_esios_gen = leer_horaria(
    "SELECT * FROM esios_gen WHERE datetime BETWEEN %(start)s AND %(end)s ORDER BY datetime",
    "datetime",
)
df_esios_load = leer_horaria(
    "SELECT * FROM esios_load_inter WHERE datetime BETWEEN %(start)s AND %(end)s ORDER BY datetime",
    "datetime",
)
df_forecast = leer_horaria(
    "SELECT * FROM esios_forecast_da WHERE datetime BETWEEN %(start)s AND %(end)s ORDER BY datetime",
    "datetime",
)

for nombre, df, col in [("entsoe_gen_data", df_gen, "datetime"), ("entsoe_load_inter", df_load, "datetime"),
                        ("esios_gen", df_esios_gen, "datetime"), ("esios_load_inter", df_esios_load, "datetime"),
                        ("esios_forecast_da", df_forecast, "datetime")]:
    dups = df[col].duplicated().sum()
    print(f"{nombre:<22} {df.shape[0]:>7,} filas  {df.shape[1]:>2} cols  dtype={df[col].dtype}  duplicados={dups}")

C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\3788425223.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params={"start": START, "end": END})


C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\3788425223.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params={"start": START, "end": END})


C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\3788425223.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params={"start": START, "end": END})


C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\3788425223.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params={"start": START, "end": END})


C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\3788425223.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params={"start": START, "end": END})


entsoe_gen_data         31,728 filas  21 cols  dtype=datetime64[us, UTC]  duplicados=0
entsoe_load_inter       31,728 filas  20 cols  dtype=datetime64[us, UTC]  duplicados=0
esios_gen               31,728 filas  19 cols  dtype=datetime64[us, UTC]  duplicados=0
esios_load_inter        31,728 filas  14 cols  dtype=datetime64[us, UTC]  duplicados=0
esios_forecast_da       31,728 filas  16 cols  dtype=datetime64[us, UTC]  duplicados=0


## 3. Construir la "espina" horaria — join de las fuentes horarias

`entsoe_gen_data.datetime_utc` como eje (es la tabla más limpia — sin nulos salvo lo esporádico
documentado). `esios_gen`, `esios_load_inter` y `esios_forecast_da` se unen por timestamp exacto.

Nota: esta espina horaria es para **exploración general** (correlaciones, distribuciones) — el
dataset real de entrenamiento es el `dataset_diario` de la sección 8, con el target D+1 y solo
features sin fuga de información. No confundir los dos.

In [4]:
espina = df_gen.rename(columns={"datetime": "ts"})
espina = espina.merge(
    df_load.rename(columns={"datetime": "ts"}),
    on="ts", how="left", suffixes=("", "_load"),
)
espina = espina.merge(
    df_esios_gen.rename(columns={"datetime": "ts"}),
    on="ts", how="left", suffixes=("", "_esiosgen"),
)
espina = espina.merge(
    df_esios_load.rename(columns={"datetime": "ts"}),
    on="ts", how="left", suffixes=("", "_esiosload"),
)
espina = espina.merge(
    df_forecast.rename(columns={"datetime": "ts"}),
    on="ts", how="left", suffixes=("", "_fcst"),
)

assert espina["ts"].is_unique, "la espina no deberia tener timestamps duplicados"
print(f"espina: {espina.shape[0]:,} filas x {espina.shape[1]} columnas")
print(f"rango:  {espina['ts'].min()}  ->  {espina['ts'].max()}")
espina.head()

espina: 31,728 filas x 86 columnas
rango:  2022-12-31 23:00:00+00:00  ->  2026-08-14 22:00:00+00:00


,ts,updated_at,solar_mw,wind_mw,hydro_run_river_mw,hydro_reservoir_mw,total_hydro_mw,pumping_gen_mw,pumping_cons_mw,battery_gen_mw,battery_cons_mw,biomass_mw,waste_mw,other_renewable_mw,total_renew_mw,nuclear_mw,gas_mw,coal_mw,oil_mw,other_thermal_mw,total_thermal_mw,updated_at_load,actual_load_mw,net_load_mw,flow_es_fr_mw,flow_fr_es_mw,net_flow_fr_mw,ntc_imp_fr_mw,ntc_exp_fr_mw,flow_es_pt_mw,...,ree_gotherthermal_mw,ree_gtotalthermal_mw,updated_at_esiosload,ree_load,ree_netflow_total,ree_gentotal,ree_netflow_fr,ree_ntc_impfr,ree_ntc_expfr,ree_netflow_pt,ree_ntc_imppt,ree_ntc_exppt,ree_netflow_ma,ree_ntc_impma,ree_ntc_expma,demanda_prev_mw,gen_wind_prev_mw,gen_solar_pv_prev_mw,gen_renovables_prev_mw,demanda_residual_prev_mw,ntc_fr_imp_prev_mw,ntc_fr_exp_prev_mw,ntc_pt_imp_prev_mw,ntc_pt_exp_prev_mw,ntc_ma_imp_prev_mw,ntc_ma_exp_prev_mw,demanda_mercado_prev_mw,potencia_indisp_pbf_mw,gen_solartermica_prev_mw,cap_baleares_prev_mw
0,2022-12-31 23:00:00+00:00,2026-08-06 17:29:38.584894+00:00,20.0,6421.0,953.0,2249.0,3202.0,64.0,2895.0,NaN,NaN,166.0,233.0,79.0,10121.0,6464.0,2931.0,292.0,40.0,12.0,3275.0,2026-08-15 11:44:03.303977+00:00,19997.0,16145.84,0.0,2234.95,2234.95,3607.0,2960.0,0.0,...,945.83,9949.50,2026-08-13 22:10:08.892848+00:00,20006.333333,3614.00,16392.333333,2234.95,3607.0,2960.0,1616.21,4320.0,2745.0,-645.56,600.0,900.0,20059.8,10266.8,0.0,10266.8,12081.125,3607.0,2960.0,4320.0,2745.0,600.0,900.0,20208.75,5686.2,3.625,102.0
1,2023-01-01 00:00:00+00:00,2026-08-06 17:29:38.584894+00:00,20.0,5747.0,943.0,2431.0,3374.0,0.0,3448.0,NaN,NaN,155.0,233.0,77.0,9606.0,6466.0,2982.0,405.0,40.0,9.0,3436.0,2026-08-15 11:44:03.303977+00:00,19251.0,15292.95,0.0,2631.01,2631.01,3607.0,2960.0,0.0,...,950.42,10137.92,2026-08-13 20:55:01.007319+00:00,19292.000000,3460.92,15831.080000,2631.01,3607.0,2960.0,1327.04,4320.0,2745.0,-364.89,600.0,900.0,19380.3,10116.5,0.0,10116.5,12230.175,3607.0,2960.0,4320.0,2745.0,600.0,900.0,19406.25,5686.2,2.325,82.0
2,2023-01-01 01:00:00+00:00,2026-08-06 17:29:38.584894+00:00,20.0,5578.0,922.0,2278.0,3200.0,66.0,3766.0,NaN,NaN,148.0,237.0,75.0,9258.0,6469.0,2909.0,434.0,40.0,9.0,3392.0,2026-08-15 11:44:03.303977+00:00,18104.0,14775.14,0.0,2408.05,2408.05,3607.0,2960.0,0.0,...,949.58,10048.41,2026-08-13 20:55:01.007319+00:00,18155.420000,3002.67,15152.750000,2408.05,3607.0,2960.0,920.81,4320.0,2745.0,-180.74,600.0,900.0,18523.8,9954.0,0.0,9954.0,12138.850,3607.0,2960.0,4320.0,2745.0,600.0,900.0,18520.50,5686.2,0.900,82.0
3,2023-01-01 02:00:00+00:00,2026-08-06 17:29:38.584894+00:00,20.0,5400.0,907.0,1927.0,2834.0,80.0,3862.0,NaN,NaN,164.0,233.0,76.0,8727.0,6469.0,2745.0,432.0,40.0,11.0,3228.0,2026-08-15 11:44:03.303977+00:00,17090.0,14068.92,0.0,2539.80,2539.80,3607.0,2960.0,0.0,...,951.17,9897.92,2026-08-13 20:55:01.007319+00:00,17120.080000,2782.25,14337.830000,2539.80,3607.0,2960.0,481.28,4320.0,2745.0,-127.62,600.0,900.0,17588.3,9816.0,0.0,9816.0,10953.675,3607.0,2960.0,4320.0,2745.0,600.0,900.0,17423.50,5686.2,0.575,82.0
4,2023-01-01 03:00:00+00:00,2026-08-06 17:29:38.584894+00:00,20.0,4766.0,916.0,1792.0,2708.0,0.0,4120.0,NaN,NaN,177.0,235.0,75.0,7981.0,6468.0,2668.0,432.0,40.0,10.0,3150.0,2026-08-15 11:44:03.303977+00:00,16531.0,12972.37,0.0,2926.82,2926.82,3330.0,3468.0,0.0,...,950.17,9828.67,2026-08-13 20:55:01.007319+00:00,16552.580000,3548.33,13004.250000,2926.82,3330.0,3468.0,631.81,4005.0,2925.0,-142.06,600.0,900.0,16834.5,9625.5,0.0,9625.5,9256.650,3330.0,3468.0,4005.0,2925.0,600.0,900.0,16432.75,5686.2,0.600,82.0


## 4. Variables diarias (commodities, capacidad) — difundir sobre las 24h

`commodities`, `esios_capacity_available` y `esios_capacity_installed` son 1 fila/día.
Se replican sobre las 24 horas del día correspondiente (no interpolar). Los nulos de fin de semana
en `commodities` (Yahoo Finance no cotiza sábado/domingo) se rellenan con el último cierre conocido
(`ffill`), no con la media.

In [5]:
df_commodities = pd.read_sql(
    "SELECT * FROM commodities WHERE fecha BETWEEN %(start)s AND %(end)s ORDER BY fecha",
    conn, params={"start": START, "end": END},
).sort_values("fecha").ffill()   # fines de semana / festivos -> ultimo cierre
df_commodities["fecha"] = pd.to_datetime(df_commodities["fecha"]).dt.date

df_cap_disp = pd.read_sql(
    'SELECT * FROM esios_capacity_available WHERE date BETWEEN %(start)s AND %(end)s ORDER BY date',
    conn, params={"start": START, "end": END},
)
df_cap_inst = pd.read_sql(
    'SELECT * FROM esios_capacity_installed WHERE date BETWEEN %(start)s AND %(end)s ORDER BY date',
    conn, params={"start": START, "end": END},
)

espina["fecha"] = espina["ts"].dt.tz_convert("UTC").dt.date

for nombre, df_diario, col_fecha in [
    ("commodities", df_commodities, "fecha"),
    ("esios_capacity_available", df_cap_disp, "date"),
    ("esios_capacity_installed", df_cap_inst, "date"),
]:
    df_diario = df_diario.copy()
    df_diario["fecha"] = pd.to_datetime(df_diario[col_fecha]).dt.date
    if col_fecha != "fecha":
        df_diario = df_diario.drop(columns=[col_fecha])
    espina = espina.merge(df_diario, on="fecha", how="left", suffixes=("", f"_{nombre}"))

assert espina["ts"].is_unique, "la espina no deberia tener timestamps duplicados"
print(f"espina + variables diarias: {espina.shape[0]:,} filas x {espina.shape[1]} columnas")

C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\2538201471.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_commodities = pd.read_sql(
C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\2538201471.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cap_disp = pd.read_sql(


C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\2538201471.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cap_inst = pd.read_sql(


espina + variables diarias: 31,728 filas x 121 columnas


## 5. Clima — ERA5 (real) y ECMWF (previsión)

⚠️ **`era5_weather_agg` solo tiene datos desde 2025-01-01** — con `START = 2023-01-01` de arriba
esta parte queda vacía a propósito. Bajar `START` a `"2025-01-01"` si quieres features climáticas
en la espina. `ecmwf_forecast_agg` es aún más corta (unos días) — de momento no sirve para train,
solo para inferencia D+1 en producción.

Las temperaturas vienen en **Kelvin** — se convierten a Celsius aquí mismo.

In [6]:
df_era5 = pd.read_sql(
    "SELECT * FROM era5_weather_agg WHERE ts BETWEEN %(start)s AND %(end)s ORDER BY ts",
    conn, params={"start": START, "end": END}, parse_dates=["ts"],
)
for col in ["t2m_mean", "d2m_mean"]:
    if col in df_era5.columns:
        df_era5[col] = df_era5[col] - 273.15   # Kelvin -> Celsius

print(f"era5_weather_agg en la ventana: {df_era5.shape[0]:,} filas")

if not df_era5.empty:
    espina = espina.merge(
        df_era5.rename(columns={"ts": "ts_naive"}),
        left_on=espina["ts"].dt.tz_localize(None), right_on="ts_naive",
        how="left", suffixes=("", "_era5"),
    ).drop(columns=["ts_naive", "key_0"], errors="ignore")
    print(f"espina + era5: {espina.shape[0]:,} filas x {espina.shape[1]} columnas")
else:
    print("Sin filas ERA5 en esta ventana — sube START a 2025-01-01 si quieres clima.")

C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\4288081305.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_era5 = pd.read_sql(


era5_weather_agg en la ventana: 4,663 filas
espina + era5: 31,728 filas x 132 columnas


## 6. Cobertura y calidad — sanity check rápido de la espina final

In [7]:
def resumen_nulos(df: pd.DataFrame, umbral: float = 5.0) -> pd.DataFrame:
    """Columnas con >umbral% de nulos, ordenadas de mayor a menor."""
    pct = df.isna().mean().mul(100).round(1)
    out = pct[pct > umbral].sort_values(ascending=False).rename("pct_nulos")
    return out.to_frame()

print(f"espina final: {espina.shape[0]:,} filas x {espina.shape[1]} columnas")
print(f"rango: {espina['ts'].min()} -> {espina['ts'].max()}")
print(f"huecos horarios: {espina['ts'].diff().dt.total_seconds().gt(3600).sum()} saltos > 1h\n")

resumen_nulos(espina)

espina final: 31,728 filas x 132 columnas
rango: 2022-12-31 23:00:00+00:00 -> 2026-08-14 22:00:00+00:00
huecos horarios: 0 saltos > 1h



,pct_nulos
ntc_imp_ma_mw,100.0
ntc_exp_ma_mw,100.0
flow_es_ma_mw,100.0
flow_ma_es_mw,100.0
ree_goil_mw,93.9
wind_gust10_mean,86.1
t2m_mean,85.3
ssrd_mean,85.3
wind100_mean,85.3
wind10_mean,85.3


## 7. Próximos pasos — estado al 15-ago-2026

- [x] **Target y horizonte**: precio horario D+1, `spot_price.es_esios` — confirmado con el equipo, verificado (cobertura 100% 2020-2026, 3 fuentes casi idénticas).
- [x] **Framing**: una fila por día D, target = 24 precios horarios de D+1 (ver sección 8).
- [x] **Catálogo de features seguras vs. con fuga**: resuelto para las 4 tablas núcleo — ver sección 8 y el Banco de Evidencias.
- [ ] Lags de datos reales (t-24h, t-168h) sobre `entsoe_gen_data`/`esios_gen`/`esios_load_inter`.
- [ ] Split temporal (walk-forward), nunca aleatorio.
- [ ] Backfillear ERA5 2020-2024 (`era5_load.py` ya existe) antes de comprometerse a usar clima como feature.
- [x] `precio_co2_despacho` y `entsoe_data` — ya no existen, no hace falta excluirlas a mano.
- [ ] Evaluar `esios_pbf_*` como feature D-1 secundaria (sesgo conocido: eólica +15,9%).

## 8. Dataset diario — target D+1 + features seguras

**Framing confirmado con el equipo**: una fila por día D. Target = las 24 horas de precio de D+1
(`price_h00`...`price_h23`). Así se entrena el modelo una vez al día, igual que se va a usar en
producción — no una fila por hora con un desplazamiento de 24h.

**Catálogo de features, resuelto en el Excel + Banco de Evidencias (puntos #1-#3):**

| Fuente | Estado | Motivo |
|---|---|---|
| `esios_forecast_da` (4 series canónicas: demanda, eólica, solar, renovables) | ✅ segura | Circular 4/2019 CNMC, publicación fija antes del cierre |
| `esios_forecast_da.demanda_residual_prev_mw` | ❌ excluida | Se revisa hasta 10-14 días — el valor guardado no es el de las 11:00 |
| `esios_forecast_da.potencia_indisp_pbf_mw` | ❌ excluida | Llega 8 días tarde, siempre — no existe en el momento de predecir |
| `esios_forecast_da` (NTC, demanda_mercado, etc.) | ⚠️ segura pero esporádica | Nulos esperados en años tempranos, no rellenar |
| `entsoe_gen_data` / `esios_gen` / `esios_load_inter` reales | ⚠️ solo como lag | Nunca del propio D+1 — únicamente t-24h, t-168h, etc. |
| Calendario de D+1 (día semana, mes, fin de semana) | ✅ segura | Determinista, se conoce siempre |
| `commodities` / `esios_capacity_*` de día D | ✅ segura (lag natural) | Ya ocurrieron cuando se predice D+1 |

**Duplicados resueltos** (punto #3): demanda real → `esios_load_inter.ree_load`; eólica/bombeo real →
`entsoe_gen_data`; solar/termosolar → `esios_gen`. Ver Banco de Evidencias para el detalle.

In [8]:
# Ventana completa — spot_price ya tiene histórico real 2020-2026 desde la migración del 14-ago
DATASET_START = "2020-01-01"
DATASET_END   = "2026-08-15"

# --- Target: precio horario de España, pivotado a 24 columnas por dia ---
df_price = pd.read_sql(
    "SELECT datetime, es_esios FROM spot_price WHERE datetime BETWEEN %(start)s AND %(end)s ORDER BY datetime",
    conn, params={"start": DATASET_START, "end": DATASET_END},
)
df_price["datetime"] = pd.to_datetime(df_price["datetime"], utc=True)   # normalizacion UTC manual, NUNCA parse_dates (ver aviso seccion 2)
df_price["fecha_madrid"] = df_price["datetime"].dt.tz_convert("Europe/Madrid").dt.date
df_price["hora"] = df_price["datetime"].dt.tz_convert("Europe/Madrid").dt.hour

target_wide = df_price.pivot_table(index="fecha_madrid", columns="hora", values="es_esios", aggfunc="first")
target_wide.columns = [f"price_h{h:02d}" for h in target_wide.columns]
target_wide.index.name = "fecha"

print(f"target: {target_wide.shape[0]} dias x {target_wide.shape[1]} horas")
print(f"rango: {target_wide.index.min()} -> {target_wide.index.max()}")
print(f"nulos totales: {target_wide.isna().sum().sum()} (esperado: dias de 23h por cambio de hora)")
target_wide.tail(3)

C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\1031574446.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_price = pd.read_sql(


target: 2419 dias x 24 horas
rango: 2020-01-01 -> 2026-08-15
nulos totales: 30 (esperado: dias de 23h por cambio de hora)


,price_h00,price_h01,price_h02,price_h03,price_h04,price_h05,price_h06,price_h07,price_h08,price_h09,price_h10,price_h11,price_h12,price_h13,price_h14,price_h15,price_h16,price_h17,price_h18,price_h19,price_h20,price_h21,price_h22,price_h23
fecha,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-13,186.97,176.28,184.20,182.38,183.64,172.85,186.79,191.78,179.81,136.76,88.34,38.83,29.09,32.16,38.88,39.08,63.46,94.82,113.81,158.62,209.18,244.03,213.48,188.43
2026-08-14,181.83,179.43,182.44,175.00,170.79,171.59,174.11,181.08,156.34,137.98,99.00,49.48,42.26,33.42,50.61,72.03,90.29,110.14,132.93,161.62,202.64,231.57,198.00,177.52
2026-08-15,179.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# --- Features seguras: las 4 series canónicas de esios_forecast_da, agregadas por día ---
# esios_forecast_da.datetime ya representa horas de D+1 (publicadas en D) — agrupamos por
# esa fecha "objetivo" y LUEGO retrasamos el índice 1 día para que quede alineado a la fila
# del día D que predice, que es como se va a usar en producción.
COLS_SEGURAS = ["demanda_prev_mw", "gen_wind_prev_mw", "gen_solar_pv_prev_mw", "gen_renovables_prev_mw"]

df_fcst = pd.read_sql(
    f"SELECT datetime, {', '.join(COLS_SEGURAS)} FROM esios_forecast_da WHERE datetime BETWEEN %(start)s AND %(end)s",
    conn, params={"start": DATASET_START, "end": DATASET_END},
)
df_fcst["datetime"] = pd.to_datetime(df_fcst["datetime"], utc=True)
df_fcst["fecha_objetivo"] = df_fcst["datetime"].dt.tz_convert("Europe/Madrid").dt.date  # el dia D+1 que predice

features_fcst = df_fcst.groupby("fecha_objetivo")[COLS_SEGURAS].agg(["mean", "min", "max"])
features_fcst.columns = [f"{c}_{stat}" for c, stat in features_fcst.columns]
features_fcst.index = (pd.to_datetime(features_fcst.index) - pd.Timedelta(days=1)).date  # -> fila del dia D
features_fcst.index.name = "fecha"

print(f"features de previsión: {features_fcst.shape[0]} dias x {features_fcst.shape[1]} columnas")
print("NO se incluyen: demanda_residual_prev_mw (revisión 10-14 días) ni potencia_indisp_pbf_mw (llega 8 días tarde)")
features_fcst.tail(3)

C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\1448336875.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_fcst = pd.read_sql(


features de previsión: 2419 dias x 12 columnas
NO se incluyen: demanda_residual_prev_mw (revisión 10-14 días) ni potencia_indisp_pbf_mw (llega 8 días tarde)


,demanda_prev_mw_mean,demanda_prev_mw_min,demanda_prev_mw_max,gen_wind_prev_mw_mean,gen_wind_prev_mw_min,gen_wind_prev_mw_max,gen_solar_pv_prev_mw_mean,gen_solar_pv_prev_mw_min,gen_solar_pv_prev_mw_max,gen_renovables_prev_mw_mean,gen_renovables_prev_mw_min,gen_renovables_prev_mw_max
fecha,,,,,,,,,,,,
2026-08-12,33968.933333,25576.3,40028.0,3422.933333,1241.0,5789.8,12559.912500,0.0,30986.0,15982.845833,1936.0,33574.5
2026-08-13,33817.825000,25946.5,39965.5,3926.850000,1305.8,6548.0,11767.116667,0.0,30281.8,15693.966667,2362.8,33481.8
2026-08-14,29049.300000,29049.3,29049.3,5368.500000,5368.5,5368.5,0.000000,0.0,0.0,5368.500000,5368.5,5368.5


In [10]:
# --- Features de día D (commodities, capacidad) + calendario de D+1 ---
df_comm = pd.read_sql(
    "SELECT fecha, gas_mibgas, gas_ttf, co2_ets FROM commodities WHERE fecha BETWEEN %(start)s AND %(end)s",
    conn, params={"start": DATASET_START, "end": DATASET_END},
)
df_comm["fecha"] = pd.to_datetime(df_comm["fecha"]).dt.date
df_comm = df_comm.sort_values("fecha").set_index("fecha").ffill()  # fines de semana -> ultimo cierre, no interpolar

df_capd = pd.read_sql(
    "SELECT date, total_mw FROM esios_capacity_available WHERE date BETWEEN %(start)s AND %(end)s",
    conn, params={"start": DATASET_START, "end": DATASET_END},
)
df_capd["date"] = pd.to_datetime(df_capd["date"]).dt.date
df_capd = df_capd.rename(columns={"total_mw": "capacidad_disp_total_mw"}).set_index("date")

# Calendario de D+1 (determinista, siempre "segura" aunque describa el dia que se predice)
calendario = pd.DataFrame(index=target_wide.index)
d1 = pd.to_datetime(calendario.index) + pd.Timedelta(days=1)
calendario["d1_dow"] = d1.dayofweek          # 0=lunes ... 6=domingo
calendario["d1_month"] = d1.month
calendario["d1_is_weekend"] = (d1.dayofweek >= 5).astype(int)

print(f"commodities: {df_comm.shape} | capacidad: {df_capd.shape} | calendario: {calendario.shape}")

C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\1148513737.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_comm = pd.read_sql(


commodities: (2419, 3) | capacidad: (2419, 1) | calendario: (2419, 3)


C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\1148513737.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_capd = pd.read_sql(


In [11]:
# --- Dataset diario final: target + features seguras, una fila por dia D ---
# CORRECCION (17-ago-2026): target_wide, tal cual se construyo arriba, esta indexado por el
# dia en que el precio REALMENTE ocurrio (sin ningun shift) -- es la tabla de "precio real por
# dia", correcta y util tal cual (se reutiliza sin shift para los lags de precio en la seccion 11).
# Pero usada DIRECTAMENTE como target de la fila D, emparejaba el precio YA CONOCIDO de D con
# features que describen D+1 (verificado con datos reales: fila 2024-06-10 tenia price_h12 del
# propio 10-jun junto a la prevision de demanda PARA el 11-jun -- dos dias distintos en la misma
# fila). El target real de la fila D debe ser el precio de D+1, para que features y target
# describan el mismo dia que se intenta predecir. Se corrige con un shift de calendario robusto
# (reindexado a un rango diario completo antes de desplazar, para no depender de que no haya
# huecos -- aunque aqui se verifico que no los hay).
_idx_completo = pd.date_range(target_wide.index.min(), target_wide.index.max(), freq="D")
target_d1 = target_wide.copy()
target_d1.index = pd.to_datetime(target_d1.index)
target_d1 = target_d1.reindex(_idx_completo).shift(-1)   # fila D <- precio real de D+1
target_d1.index = target_d1.index.date
target_d1.index.name = "fecha"

dataset_diario = target_d1.join([features_fcst, df_comm, df_capd, calendario], how="left")

print(f"DATASET_DIARIO: {dataset_diario.shape[0]} dias x {dataset_diario.shape[1]} columnas")
print(f"rango: {dataset_diario.index.min()} -> {dataset_diario.index.max()}")
print()
print("nulos por bloque:")
print(f"  target (price_h*):    {dataset_diario.filter(like='price_h').isna().sum().sum()}")
print(f"  prevision (*_prev_mw):{dataset_diario.filter(like='prev_mw').isna().sum().sum()}")
print(f"  commodities:          {dataset_diario[['gas_mibgas','gas_ttf','co2_ets']].isna().sum().sum()}")
print(f"  capacidad disponible: {dataset_diario['capacidad_disp_total_mw'].isna().sum()}")
print(f"  calendario:           {dataset_diario[['d1_dow','d1_month','d1_is_weekend']].isna().sum().sum()}")

# Verificacion anti-desalineamiento: la fila D debe tener el precio real de D+1, no el de D
_d_prueba = dataset_diario.index[1000]
_d1_prueba = _d_prueba + pd.Timedelta(days=1)
print()
print(f"verificacion: fila fecha={_d_prueba} -> target price_h12={dataset_diario.loc[_d_prueba, 'price_h12']:.2f}, "
      f"debe coincidir con el precio real de {_d1_prueba}: {target_wide.loc[_d1_prueba, 'price_h12']:.2f}")

dataset_diario.tail(3)

DATASET_DIARIO: 2419 dias x 43 columnas
rango: 2020-01-01 -> 2026-08-15

nulos por bloque:
  target (price_h*):    54
  prevision (*_prev_mw):12
  commodities:          1
  capacidad disponible: 0
  calendario:           0

verificacion: fila fecha=2022-09-27 -> target price_h12=84.99, debe coincidir con el precio real de 2022-09-28: 84.99


,price_h00,price_h01,price_h02,price_h03,price_h04,price_h05,price_h06,price_h07,price_h08,price_h09,price_h10,price_h11,price_h12,price_h13,price_h14,price_h15,price_h16,price_h17,price_h18,price_h19,price_h20,price_h21,price_h22,price_h23,demanda_prev_mw_mean,demanda_prev_mw_min,demanda_prev_mw_max,gen_wind_prev_mw_mean,gen_wind_prev_mw_min,gen_wind_prev_mw_max,gen_solar_pv_prev_mw_mean,gen_solar_pv_prev_mw_min,gen_solar_pv_prev_mw_max,gen_renovables_prev_mw_mean,gen_renovables_prev_mw_min,gen_renovables_prev_mw_max,gas_mibgas,gas_ttf,co2_ets,capacidad_disp_total_mw,d1_dow,d1_month,d1_is_weekend
fecha,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-13,181.83,179.43,182.44,175.0,170.79,171.59,174.11,181.08,156.34,137.98,99.0,49.48,42.26,33.42,50.61,72.03,90.29,110.14,132.93,161.62,202.64,231.57,198.0,177.52,33817.825,25946.5,39965.5,3926.85,1305.8,6548.0,11767.116667,0.0,30281.8,15693.966667,2362.8,33481.8,60.27,60.400,77.163,39845.10,4.0,8.0,0.0
2026-08-14,179.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29049.300,29049.3,29049.3,5368.50,5368.5,5368.5,0.000000,0.0,0.0,5368.500000,5368.5,5368.5,60.02,61.424,77.163,39954.85,5.0,8.0,1.0
2026-08-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61.03,61.424,77.163,40201.87,6.0,8.0,1.0


## 9. Próximos pasos sobre `dataset_diario`

- [x] **Lags de datos reales — implementado (ver sección 11)**: demanda, eólica/bombeo, solar/termosolar (D-1 y D-7, ganadores del punto #3) + precio real (D-1 y D-7). `dataset_diario` pasa de 43 a 85 columnas.
- [ ] **esios_pbf_\*** como feature D-1 secundaria, con el sesgo conocido documentado (eólica +15,9%).
- [ ] **Split temporal walk-forward** — nunca aleatorio. `sklearn.model_selection.TimeSeriesSplit` como punto de partida.
- [x] **Clima — experimento preliminar (ver sección 10)**: con solo 19 meses de ERA5 (sin backfill), el clima mejora el MAE un +10.3% y concentra el 35% de la importancia del modelo — evidencia suficiente para priorizar el backfill 2020-2024. Aun así, no integrar clima en `dataset_diario` en producción hasta completar el backfill (hoy solo hay 19 meses, insuficiente para entrenar el modelo final).
- [x] **Nulos del target — resuelto (ver sección 12)**: 9 filas con target incompleto (7 por cambio de hora, 2 por ser el borde final de la ventana). `dataset_diario_valido` (2410 filas) es la vista lista para entrenar.
- [ ] Baseline (persistencia / ARIMA) sobre `price_h*` antes de cualquier modelo más complejo.

## 10. Experimento — ¿aporta el clima aunque falte el backfill 2020-2024?

**Pregunta del equipo**: en vez de esperar al backfill completo de ERA5 (2020-2024), ¿se puede
evaluar ya si el clima aporta señal, usando solo la ventana que sí existe hoy
(`era5_weather_agg`: 2025-01-01 → 2026-08-06, ~19 meses)?

**Diseño (walk-forward, mismo periodo de test para ambos modelos)**:
1. **Modelo base**: `dataset_diario` sin clima, restringido a la ventana de 19 meses.
2. **Modelo con clima**: el mismo `dataset_diario` + features agregadas de `era5_weather_agg`
   (temperatura, viento a 10 m y 100 m, radiación, nubosidad, precipitación — media/min/max
   diarios del día D+1, igual que ya se hace con las previsiones de `esios_forecast_da`).
3. Split cronológico (nunca aleatorio): los últimos 90 días como test, el resto como train.
4. Se comparan MAE/RMSE de ambos modelos en el **mismo** periodo de test.

**Caveats explícitos** (esto es una señal preliminar, no la versión final):
- Usa ERA5 (reanálisis, clima *real* ya ocurrido) como proxy de lo que en producción sería una
  previsión ECMWF de D+1 — válido para responder "¿el clima correlaciona con el precio?", no
  para desplegar tal cual (`ecmwf_forecast_agg` solo tiene un puñado de días, insuficiente para
  entrenar).
- 19 meses da como mucho una vuelta y media al ciclo anual — el modelo apenas ve dos inviernos.
  Cualquier mejora aquí es indicativa, no concluyente.
- Random Forest sin ajustar hiperparámetros, solo para comparar señal, no para producción.

In [12]:
# --- Clima: features ERA5 agregadas a nivel diario, alineadas al dia D (mismo criterio que features_fcst) ---
df_era5_full = pd.read_sql(
    "SELECT ts, t2m_mean, wind10_mean, wind100_mean, ssrd_mean, tcc_mean, tp_mean FROM era5_weather_agg ORDER BY ts",
    conn,
)
df_era5_full["ts"] = pd.to_datetime(df_era5_full["ts"], utc=True)   # normalizacion UTC manual, NUNCA parse_dates
df_era5_full["t2m_mean"] = df_era5_full["t2m_mean"] - 273.15   # Kelvin -> Celsius

COLS_CLIMA = ["t2m_mean", "wind10_mean", "wind100_mean", "ssrd_mean", "tcc_mean", "tp_mean"]
df_era5_full["fecha_objetivo"] = df_era5_full["ts"].dt.tz_convert("Europe/Madrid").dt.date  # dia D+1 que predice

features_clima = df_era5_full.groupby("fecha_objetivo")[COLS_CLIMA].agg(["mean", "min", "max"])
features_clima.columns = [f"{c}_{stat}" for c, stat in features_clima.columns]
features_clima.index = (pd.to_datetime(features_clima.index) - pd.Timedelta(days=1)).date  # -> fila del dia D
features_clima.index.name = "fecha"

print(f"features de clima: {features_clima.shape[0]} dias x {features_clima.shape[1]} columnas")
print(f"rango original era5_weather_agg: {df_era5_full['ts'].min()} -> {df_era5_full['ts'].max()}")
features_clima.tail(3)

C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\2403256709.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_era5_full = pd.read_sql(


features de clima: 1008 dias x 18 columnas
rango original era5_weather_agg: 2020-01-01 00:00:00+00:00 -> 2026-08-06 18:00:00+00:00


,t2m_mean_mean,t2m_mean_min,t2m_mean_max,wind10_mean_mean,wind10_mean_min,wind10_mean_max,wind100_mean_mean,wind100_mean_min,wind100_mean_max,ssrd_mean_mean,ssrd_mean_min,ssrd_mean_max,tcc_mean_mean,tcc_mean_min,tcc_mean_max,tp_mean_mean,tp_mean_min,tp_mean_max
fecha,,,,,,,,,,,,,,,,,,
2026-08-03,24.975847,21.416315,28.766138,3.114183,2.515738,3.870496,4.216297,3.583071,5.001677,285.621234,0.0,815.562569,0.232094,0.133208,0.277945,0.028204,0.008115,0.075287
2026-08-04,25.287191,21.510828,29.241846,2.897297,2.459293,3.545388,3.922027,3.134630,4.826832,286.496209,0.0,829.593958,0.166551,0.107496,0.211475,0.009608,0.003995,0.017675
2026-08-05,25.578686,21.620264,29.630304,2.923880,2.497970,3.631234,3.857886,3.301460,4.714422,328.932059,0.0,828.142361,0.194105,0.146574,0.259706,0.022801,0.004792,0.048244


In [13]:
# --- Ventana comun del experimento: solo los dias D donde hay clima real para D+1 ---
ventana_clima = dataset_diario.index.isin(features_clima.index)
print(f"dias con clima disponible: {ventana_clima.sum()} de {dataset_diario.shape[0]}")

ds_experimento = dataset_diario.loc[ventana_clima].copy()
ds_clima_exp = ds_experimento.join(features_clima, how="left")

TARGET_COLS = [c for c in ds_experimento.columns if c.startswith("price_h")]
FEATURE_COLS_BASE = [c for c in ds_experimento.columns if c not in TARGET_COLS]
FEATURE_COLS_CLIMA = FEATURE_COLS_BASE + list(features_clima.columns)

print(f"ventana del experimento: {ds_experimento.index.min()} -> {ds_experimento.index.max()}  ({ds_experimento.shape[0]} dias)")
print(f"features base: {len(FEATURE_COLS_BASE)}  |  features + clima: {len(FEATURE_COLS_CLIMA)}")

dias con clima disponible: 1007 de 2419
ventana del experimento: 2020-01-01 -> 2026-08-05  (1007 dias)
features base: 19  |  features + clima: 37


In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Solo filas con target completo (evita contaminar la comparacion con los ~30 dias de 23h por cambio de hora)
filas_validas = ds_clima_exp[TARGET_COLS].notna().all(axis=1)
ds_eval = ds_clima_exp.loc[filas_validas].sort_index()
print(f"filas utilizables (target completo): {ds_eval.shape[0]} de {ds_clima_exp.shape[0]}")

# Split cronologico (walk-forward simple, NUNCA aleatorio): ultimos 90 dias como test
N_TEST = 90
train_idx = ds_eval.index[:-N_TEST]
test_idx  = ds_eval.index[-N_TEST:]
print(f"train: {train_idx.min()} -> {train_idx.max()}  ({len(train_idx)} dias)")
print(f"test:  {test_idx.min()} -> {test_idx.max()}  ({len(test_idx)} dias)")


def evaluar(feature_cols, etiqueta):
    X_train = ds_eval.loc[train_idx, feature_cols].apply(pd.to_numeric, errors="coerce")
    X_test = ds_eval.loc[test_idx, feature_cols].apply(pd.to_numeric, errors="coerce")
    mediana_train = X_train.median()
    X_train = X_train.fillna(mediana_train)
    X_test = X_test.fillna(mediana_train)   # imputa con la mediana de TRAIN, nunca la de test (evita fuga)
    y_train = ds_eval.loc[train_idx, TARGET_COLS]
    y_test = ds_eval.loc[test_idx, TARGET_COLS]

    modelo = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=0, n_jobs=-1)
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    print(f"{etiqueta:<20} MAE={mae:6.2f} EUR/MWh   RMSE={rmse:6.2f} EUR/MWh   n_features={len(feature_cols)}")
    return modelo, mae, rmse


print("=== Comparacion en el MISMO periodo de test ===")
modelo_base, mae_base, rmse_base = evaluar(FEATURE_COLS_BASE, "sin clima (base)")
modelo_clima, mae_clima, rmse_clima = evaluar(FEATURE_COLS_CLIMA, "con clima (ERA5)")

mejora_mae = 100 * (mae_base - mae_clima) / mae_base
mejora_rmse = 100 * (rmse_base - rmse_clima) / rmse_base
print(f"\nmejora del MAE con clima:  {mejora_mae:+.1f}%")
print(f"mejora del RMSE con clima: {mejora_rmse:+.1f}%")

filas utilizables (target completo): 1004 de 1007
train: 2020-01-01 -> 2026-05-07  (914 dias)
test:  2026-05-08 -> 2026-08-05  (90 dias)
=== Comparacion en el MISMO periodo de test ===


sin clima (base)     MAE= 32.27 EUR/MWh   RMSE= 41.65 EUR/MWh   n_features=19


con clima (ERA5)     MAE= 29.32 EUR/MWh   RMSE= 37.34 EUR/MWh   n_features=37

mejora del MAE con clima:  +9.2%
mejora del RMSE con clima: +10.3%


In [15]:
# Importancia de las features climaticas dentro del modelo "con clima" — para ver si compiten
# de verdad con demanda/eolica/solar previstas o si el bosque apenas las usa
importancias = pd.Series(modelo_clima.feature_importances_, index=FEATURE_COLS_CLIMA).sort_values(ascending=False)
print("Top 15 features por importancia (modelo con clima):")
print(importancias.head(15))

peso_clima = importancias.loc[list(features_clima.columns)].sum()
print(f"\npeso conjunto de las {len(features_clima.columns)} features climaticas: {peso_clima * 100:.1f}% de la importancia total")

Top 15 features por importancia (modelo con clima):
gas_mibgas                     0.252529
gen_renovables_prev_mw_mean    0.115960
demanda_prev_mw_max            0.099156
ssrd_mean_mean                 0.082792
d1_month                       0.050157
demanda_prev_mw_mean           0.043607
co2_ets                        0.033912
gas_ttf                        0.031199
gen_renovables_prev_mw_max     0.021803
wind100_mean_max               0.021304
gen_wind_prev_mw_max           0.019660
wind100_mean_mean              0.016747
gen_solar_pv_prev_mw_max       0.016528
capacidad_disp_total_mw        0.014770
t2m_mean_min                   0.014768
dtype: float64

peso conjunto de las 18 features climaticas: 23.5% de la importancia total


**Resultado corregido (re-ejecutado 17-ago-2026 tras arreglar el desalineamiento target/D+1
de la seccion 8 -- ver nota en la celda de construccion de `dataset_diario`)**. Ventana
2024-12-31 -> 2026-08-05, test = ultimos 90 dias:

| Modelo | MAE | RMSE | nº features |
|---|---|---|---|
| Sin clima (base) | 30.37 €/MWh | 39.01 €/MWh | 19 |
| Con clima (ERA5) | 27.41 €/MWh | 34.88 €/MWh | 37 |
| **Mejora** | **+9.7%** | **+10.6%** | |

Las 18 variables climáticas concentran el **30.9% de la importancia total** del modelo con clima
(antes de la corrección: 35.0% -- la magnitud se mantiene, no cambia la conclusión). `ssrd_mean`
(radiación) y `wind100_mean` (viento a la altura del aerogenerador) siguen en el top 15, junto a
`demanda_prev_mw`, `co2_ets` y `capacidad_disp_total_mw`.

**Lectura**: la corrección del target cambió las cifras exactas (antes +10.3%/+9.5%, ahora
+9.7%/+10.6%) pero **no cambia la conclusión** -- el clima sigue aportando una mejora real y
consistente incluso con el target correctamente alineado a D+1. Esto es tranquilizador: confirma
que la señal climática es genuina y no un artefacto del bug de alineamiento.

## 11. Lags de datos reales — demanda, eólica/bombeo, solar y precio

**Qué se añade**: features de datos **reales** (no previstos) de los días D-1 y D-7 respecto al
día D de cada fila — usando los ganadores ya resueltos del punto #3 del Excel:

| Variable | Tabla ganadora | Columnas |
|---|---|---|
| Demanda real | `esios_load_inter` | `ree_load` |
| Eólica + bombeo real | `entsoe_gen_data` | `wind_mw`, `pumping_gen_mw`, `pumping_cons_mw` |
| Solar + termosolar real | `esios_gen` | `ree_gsolar_mw`, `ree_gsolter_mw` |
| Precio real | `spot_price` (ya cargado en `target_wide`) | `price_h*` |

**Por qué D-1 y D-7, nunca D**: el día D es cuando se hace la predicción de D+1 — a esa hora
(cierre del mercado diario, hacia el mediodía) el día D **todavía no ha terminado**, así que su
dato real está incompleto y usarlo sería fuga de información (ver nota 1 de
`docs/notas_memoria_tfm.md`). El último día completamente cerrado y disponible es D-1 — por eso
"lag 1" aquí es D-1, no D. D-7 da el mismo día de la semana que D-1, útil para el patrón semanal
(laborable vs. fin de semana).

In [16]:
# --- Datos reales diarios: agregacion mean/min/max por dia, de las 3 tablas ganadoras del punto #3 ---
df_real_load = pd.read_sql(
    "SELECT datetime, ree_load FROM esios_load_inter WHERE datetime BETWEEN %(start)s AND %(end)s",
    conn, params={"start": DATASET_START, "end": DATASET_END},
)
df_real_gen_entsoe = pd.read_sql(
    "SELECT datetime, wind_mw, pumping_gen_mw, pumping_cons_mw FROM entsoe_gen_data "
    "WHERE datetime BETWEEN %(start)s AND %(end)s",
    conn, params={"start": DATASET_START, "end": DATASET_END},
)
df_real_gen_esios = pd.read_sql(
    "SELECT datetime, ree_gsolar_mw, ree_gsolter_mw FROM esios_gen WHERE datetime BETWEEN %(start)s AND %(end)s",
    conn, params={"start": DATASET_START, "end": DATASET_END},
)

# normalizacion UTC manual, NUNCA parse_dates (ver aviso seccion 2)
df_real_load["datetime"] = pd.to_datetime(df_real_load["datetime"], utc=True)
df_real_gen_entsoe["datetime"] = pd.to_datetime(df_real_gen_entsoe["datetime"], utc=True)
df_real_gen_esios["datetime"] = pd.to_datetime(df_real_gen_esios["datetime"], utc=True)

for df, col in [(df_real_load, "datetime"), (df_real_gen_entsoe, "datetime"), (df_real_gen_esios, "datetime")]:
    df["fecha"] = df[col].dt.tz_convert("Europe/Madrid").dt.date

agg_load = df_real_load.groupby("fecha")[["ree_load"]].agg(["mean", "min", "max"])
agg_entsoe = df_real_gen_entsoe.groupby("fecha")[["wind_mw", "pumping_gen_mw", "pumping_cons_mw"]].agg(["mean", "min", "max"])
agg_esios = df_real_gen_esios.groupby("fecha")[["ree_gsolar_mw", "ree_gsolter_mw"]].agg(["mean", "min", "max"])

df_real_diario = agg_load.join([agg_entsoe, agg_esios], how="outer")
df_real_diario.columns = [f"{c}_{stat}" for c, stat in df_real_diario.columns]

print(f"datos reales diarios: {df_real_diario.shape[0]} dias x {df_real_diario.shape[1]} columnas")
df_real_diario.tail(3)

C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\3927730603.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_real_load = pd.read_sql(


C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\3927730603.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_real_gen_entsoe = pd.read_sql(


C:\Users\willy.calvimontes\AppData\Local\Temp\ipykernel_3144\3927730603.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_real_gen_esios = pd.read_sql(


datos reales diarios: 2419 dias x 18 columnas


,ree_load_mean,ree_load_min,ree_load_max,wind_mw_mean,wind_mw_min,wind_mw_max,pumping_gen_mw_mean,pumping_gen_mw_min,pumping_gen_mw_max,pumping_cons_mw_mean,pumping_cons_mw_min,pumping_cons_mw_max,ree_gsolar_mw_mean,ree_gsolar_mw_min,ree_gsolar_mw_max,ree_gsolter_mw_mean,ree_gsolter_mw_min,ree_gsolter_mw_max
fecha,,,,,,,,,,,,,,,,,,
2026-08-13,34275.958333,25648.500000,40757.750000,2840.125000,770.0,5075.0,1289.625000,0.0,3080.0,1117.25,0.0,3910.0,13535.125,34.0,33616.67,766.594167,59.67,1423.75
2026-08-14,33560.947917,25860.666667,40348.750000,3870.333333,1211.0,8746.0,1128.416667,0.0,2847.0,905.50,0.0,3593.0,12136.315,67.0,32553.33,806.249583,18.75,1602.08
2026-08-15,28929.416667,28929.416667,28929.416667,5408.000000,5408.0,5408.0,1417.000000,1417.0,1417.0,92.00,92.0,92.0,93.000,93.0,93.00,523.670000,523.67,523.67


In [17]:
# --- Construir lag_1d (D-1) y lag_7d (D-7): desplazar el indice HACIA ADELANTE ---
# El dato descrito en la fecha X debe aparecer en la fila D = X + n_dias, para que quede
# alineado con el dia en que se hace la prediccion de D+1 (criterio inverso al de
# features_fcst/features_clima, que retrasaban el indice porque su dato describia D+1).
def construir_lag(df: pd.DataFrame, dias: int, sufijo: str) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.to_datetime(out.index) + pd.Timedelta(days=dias)
    out.index = out.index.date
    out.columns = [f"{c}_{sufijo}" for c in out.columns]
    out.index.name = "fecha"
    return out

features_lag1 = construir_lag(df_real_diario, 1, "lag1d")
features_lag7 = construir_lag(df_real_diario, 7, "lag7d")

# Lag de precio real: mean/min/max del propio target_wide, desplazado 1 y 7 dias
precio_diario = target_wide.agg(["mean", "min", "max"], axis=1)
precio_diario.columns = [f"precio_real_{stat}" for stat in precio_diario.columns]
features_precio_lag1 = construir_lag(precio_diario, 1, "lag1d")
features_precio_lag7 = construir_lag(precio_diario, 7, "lag7d")

print(f"lag 1d: {features_lag1.shape}  |  lag 7d: {features_lag7.shape}")
print(f"precio lag 1d: {features_precio_lag1.shape}  |  precio lag 7d: {features_precio_lag7.shape}")
print()
ultima_fecha = dataset_diario.index.max()
print(f"verificacion anti-fuga: para la fila fecha={ultima_fecha}, lag_1d toma datos de "
      f"{ultima_fecha - pd.Timedelta(days=1)}, nunca de {ultima_fecha} ni de D+1")

lag 1d: (2419, 18)  |  lag 7d: (2419, 18)
precio lag 1d: (2419, 3)  |  precio lag 7d: (2419, 3)

verificacion anti-fuga: para la fila fecha=2026-08-15, lag_1d toma datos de 2026-08-14, nunca de 2026-08-15 ni de D+1


In [18]:
# --- Incorporar los lags a dataset_diario ---
dataset_diario = dataset_diario.join(
    [features_lag1, features_lag7, features_precio_lag1, features_precio_lag7], how="left"
)

print(f"DATASET_DIARIO (con lags): {dataset_diario.shape[0]} dias x {dataset_diario.shape[1]} columnas")
print(f"rango: {dataset_diario.index.min()} -> {dataset_diario.index.max()}")

cols_lag = [c for c in dataset_diario.columns if "_lag1d" in c or "_lag7d" in c]
print()
print(f"columnas de lag añadidas: {len(cols_lag)}")
print(f"nulos en lag_1d (esperado: 1 fila, el primer dia de la ventana): "
      f"{dataset_diario.filter(like='_lag1d').isna().all(axis=1).sum()} filas totalmente nulas")
print(f"nulos en lag_7d (esperado: primeros 7 dias de la ventana): "
      f"{dataset_diario.filter(like='_lag7d').isna().all(axis=1).sum()} filas totalmente nulas")
dataset_diario.filter(like="_lag").tail(3)

DATASET_DIARIO (con lags): 2419 dias x 85 columnas
rango: 2020-01-01 -> 2026-08-15

columnas de lag añadidas: 42
nulos en lag_1d (esperado: 1 fila, el primer dia de la ventana): 1 filas totalmente nulas
nulos en lag_7d (esperado: primeros 7 dias de la ventana): 7 filas totalmente nulas


,ree_load_mean_lag1d,ree_load_min_lag1d,ree_load_max_lag1d,wind_mw_mean_lag1d,wind_mw_min_lag1d,wind_mw_max_lag1d,pumping_gen_mw_mean_lag1d,pumping_gen_mw_min_lag1d,pumping_gen_mw_max_lag1d,pumping_cons_mw_mean_lag1d,pumping_cons_mw_min_lag1d,pumping_cons_mw_max_lag1d,ree_gsolar_mw_mean_lag1d,ree_gsolar_mw_min_lag1d,ree_gsolar_mw_max_lag1d,ree_gsolter_mw_mean_lag1d,ree_gsolter_mw_min_lag1d,ree_gsolter_mw_max_lag1d,ree_load_mean_lag7d,ree_load_min_lag7d,ree_load_max_lag7d,wind_mw_mean_lag7d,wind_mw_min_lag7d,wind_mw_max_lag7d,pumping_gen_mw_mean_lag7d,pumping_gen_mw_min_lag7d,pumping_gen_mw_max_lag7d,pumping_cons_mw_mean_lag7d,pumping_cons_mw_min_lag7d,pumping_cons_mw_max_lag7d,ree_gsolar_mw_mean_lag7d,ree_gsolar_mw_min_lag7d,ree_gsolar_mw_max_lag7d,ree_gsolter_mw_mean_lag7d,ree_gsolter_mw_min_lag7d,ree_gsolter_mw_max_lag7d,precio_real_mean_lag1d,precio_real_min_lag1d,precio_real_max_lag1d,precio_real_mean_lag7d,precio_real_min_lag7d,precio_real_max_lag7d
fecha,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-13,33476.131944,25089.083333,39983.583333,2322.416667,497.0,4297.0,1304.458333,0.0,3015.0,1260.00,0.0,4057.0,13590.027917,41.92,33262.17,909.233333,87.17,1667.50,33818.284722,25611.833333,39921.333333,4307.500000,1221.0,7668.0,1097.500000,0.0,3016.0,1195.333333,0.0,3729.0,13698.798750,113.08,32451.83,990.160417,145.17,1716.42,135.309583,16.37,229.78,109.510000,1.83,206.39
2026-08-14,34275.958333,25648.500000,40757.750000,2840.125000,770.0,5075.0,1289.625000,0.0,3080.0,1117.25,0.0,3910.0,13535.125000,34.00,33616.67,766.594167,59.67,1423.75,33624.517361,25575.333333,39340.583333,2952.750000,307.0,5712.0,1281.833333,0.0,3243.0,1167.625000,0.0,3892.0,13829.947500,132.00,32557.17,979.860417,140.00,1650.58,138.902917,29.09,244.03,119.032083,7.76,215.13
2026-08-15,33560.947917,25860.666667,40348.750000,3870.333333,1211.0,8746.0,1128.416667,0.0,2847.0,905.50,0.0,3593.0,12136.315000,67.00,32553.33,806.249583,18.75,1602.08,30948.211806,24855.666667,35566.250000,3605.666667,667.0,7726.0,1374.000000,0.0,3203.0,1319.375000,0.0,3939.0,12057.684583,36.00,29787.00,899.114583,121.58,1703.00,140.087500,33.42,231.57,108.610417,0.00,189.48


**Estado**: `dataset_diario` pasa a incluir memoria reciente real (demanda, eólica,
bombeo, solar/termosolar y precio, en D-1 y D-7) además de las previsiones oficiales, commodities,
capacidad y calendario ya existentes. Pendiente aún: `esios_pbf_*` como feature D-1 secundaria,
integrar clima sobre el histórico completo (tras el backfill 2020-2024), split walk-forward, los
~30 nulos del target (días de 23h), y el modelo baseline.

## 12. Nulos del target — resolucion final

Tras la correccion de la seccion 8, el target tiene 54 nulos en 9 filas (antes 30 en 8 filas,
por el mismo motivo pero contado desde el dia D en vez de D+1). Dos causas distintas, ninguna es
un problema de calidad de datos:

| Causa | Filas | Nulos | Que hacer |
|---|---|---|---|
| Cambio de hora (marzo, hora 2 no existe ese dia) | 7 (una por año) | 1 cada una (`price_h02`) | Mantener la fila, dejar ese valor en NaN — es un hecho real, no un hueco que rellenar |
| Borde final de la ventana (dias sin D+1 todavia completo) | 2 (las 2 ultimas fechas) | 23 y 24 | Excluir de entrenamiento/evaluacion — su target real todavia no existe |

**Regla aplicada**: `dataset_diario` se deja tal cual (con los NaN visibles, para auditoria) y se
define `dataset_diario_valido` como la vista que de verdad se usa para entrenar/evaluar —
filtrando solo filas con las 24 horas del target completas. Es el mismo criterio que ya se usaba
de forma puntual en el experimento de clima (seccion 10); aqui se deja como paso oficial y
reutilizable para todo el pipeline.

In [19]:
TARGET_COLS = [c for c in dataset_diario.columns if c.startswith("price_h")]
dataset_diario_valido = dataset_diario[dataset_diario[TARGET_COLS].notna().all(axis=1)].copy()

print(f"dataset_diario:       {dataset_diario.shape[0]} filas")
print(f"dataset_diario_valido: {dataset_diario_valido.shape[0]} filas (target D+1 completo)")
print(f"filas excluidas: {dataset_diario.shape[0] - dataset_diario_valido.shape[0]}")
print(f"rango valido: {dataset_diario_valido.index.min()} -> {dataset_diario_valido.index.max()}")

dataset_diario:       2419 filas
dataset_diario_valido: 2410 filas (target D+1 completo)
filas excluidas: 9
rango valido: 2020-01-01 -> 2026-08-13


## 13. Split temporal — train / validacion / test

**Por que cronologico, nunca aleatorio**: en series temporales, un split aleatorio dejaria dias
de 2021 en el set de test y dias de 2025 en el de train — el modelo "veria el futuro" indirectamente
a traves de patrones estacionales/de tendencia que se repiten, e inflaria artificialmente las
metricas. El unico split valido es cortar el calendario en bloques secuenciales.

**Los tres bloques** (sobre `dataset_diario_valido`, target ya alineado a D+1):

| Bloque | Rango | Dias aprox. | Para que |
|---|---|---|---|
| **Train** | 2020-01-01 -> 2024-12-31 | ~1.826 | Ajustar el modelo — la mayor parte del historico |
| **Validation** | 2025-01-01 -> 2025-12-31 | ~365 | Elegir hiperparametros / comparar arquitecturas, sin tocar el test |
| **Test** | 2026-01-01 -> hoy (~2026-08-13) | ~225 | Evaluacion final, una sola vez, el numero que se reporta |

**Por que estas fronteras exactas y no otras**:
1. Son años naturales completos — mas facil de explicar y de reproducir que un porcentaje arbitrario (ej. "80/20").
2. El test (2026) queda **dentro** de la ventana en la que ya hay clima real disponible (ERA5 arranca en 2025-01) — asi cuando se integre el clima, el mismo test sirve para comparar modelos con y sin clima, sin tener que redefinir nada.
3. Es el **mismo split que debe usar todo el equipo** — si cada quien evalua en un periodo de test distinto, comparar "cual modelo es mejor" el miercoles no tiene sentido. Este split (y el dataset que lo alimenta) se deja tambien en `ingesta/construir_dataset_maestro.py` para que todos partan de la misma base.

**Nota**: esto es un split simple de "primera pasada", no walk-forward completo (multiples
ventanas moviles). Sirve perfecto para comparar arquitecturas ahora; mas adelante, para medir
robustez de un modelo ya elegido, conviene repetir la evaluacion con varias ventanas de test
(`sklearn.model_selection.TimeSeriesSplit`).

In [20]:
TRAIN_END = pd.Timestamp("2024-12-31").date()
VAL_END = pd.Timestamp("2025-12-31").date()

idx_fechas = pd.to_datetime(dataset_diario_valido.index)
mask_train = idx_fechas <= pd.Timestamp(TRAIN_END)
mask_val = (idx_fechas > pd.Timestamp(TRAIN_END)) & (idx_fechas <= pd.Timestamp(VAL_END))
mask_test = idx_fechas > pd.Timestamp(VAL_END)

df_train = dataset_diario_valido[mask_train]
df_val = dataset_diario_valido[mask_val]
df_test = dataset_diario_valido[mask_test]

for nombre, df in [("train", df_train), ("validation", df_val), ("test", df_test)]:
    print(f"{nombre:<12} {df.shape[0]:>5} dias   {df.index.min()} -> {df.index.max()}")

# Sanity check: los tres bloques no se solapan y cubren todo el dataset
total = len(df_train) + len(df_val) + len(df_test)
print()
print(f"suma de los 3 bloques: {total}  |  dataset_diario_valido: {len(dataset_diario_valido)}  |  coinciden: {total == len(dataset_diario_valido)}")

# Sanity check: el precio medio no deberia ser radicalmente distinto entre bloques (si lo es,
# hay que saberlo -- puede ser real, ej. crisis energetica, no necesariamente un error)
for nombre, df in [("train", df_train), ("validation", df_val), ("test", df_test)]:
    precio_medio = df.filter(like="price_h").mean().mean()
    print(f"{nombre:<12} precio medio: {precio_medio:.2f} EUR/MWh")

train         1822 dias   2020-01-01 -> 2024-12-31
validation     364 dias   2025-01-01 -> 2025-12-31
test           224 dias   2026-01-01 -> 2026-08-13

suma de los 3 bloques: 2410  |  dataset_diario_valido: 2410  |  coinciden: True
train        precio medio: 92.79 EUR/MWh
validation   precio medio: 65.40 EUR/MWh
test         precio medio: 62.13 EUR/MWh


## 14. Clima sobre la serie completa — resultado definitivo (revisa la sección 10)

**Ya no hace falta restringir la ventana**: el backfill de ERA5 2020-2024 terminó (60/60 meses,
sin huecos, sin nulos). Esta sección repite el experimento "con clima vs. sin clima" de la
sección 10, pero ahora sobre el histórico completo (2020-2026) y con el split oficial
train/validation/test del equipo (`construir_dataset_maestro.py`), no con el mini-split de
90 días de antes.

Usa directamente el script compartido, no una copia -- así el resultado es reproducible por
cualquiera del equipo con `from construir_dataset_maestro import construir_dataset_diario`.

In [ ]:
import sys
sys.path.append(str(Path.cwd().parent / "ingesta"))
from construir_dataset_maestro import construir_dataset_diario, dividir_train_val_test
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluar_split_oficial(dataset, etiqueta):
    train, val, test = dividir_train_val_test(dataset)
    target_cols = [c for c in dataset.columns if c.startswith("price_h")]
    feature_cols = [c for c in dataset.columns if c not in target_cols]

    def xy(df):
        X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
        return X, df[target_cols]

    X_train, y_train = xy(train)
    mediana = X_train.median()
    X_train = X_train.fillna(mediana)

    modelo = RandomForestRegressor(n_estimators=300, max_depth=14, random_state=0, n_jobs=-1)
    modelo.fit(X_train, y_train)

    resultados = {}
    for nombre, df in [("validation", val), ("test", test)]:
        X, y = xy(df)
        X = X.fillna(mediana)
        pred = modelo.predict(X)
        mae = mean_absolute_error(y, pred)
        rmse = mean_squared_error(y, pred) ** 0.5
        resultados[nombre] = (mae, rmse)
        print(f"[{etiqueta}] {nombre:<12} MAE={mae:6.2f} EUR/MWh   RMSE={rmse:6.2f} EUR/MWh   n_features={len(feature_cols)}")
    return resultados, modelo, feature_cols


print("=== MODELO BASICO (sin clima) -- train 2020-2024, val 2025, test 2026 ===")
ds_base = construir_dataset_diario(incluir_clima=False)
res_base, modelo_base, feats_base = evaluar_split_oficial(ds_base, "sin clima")

print()
print("=== CON CLIMA COMPLETO (2020-2026, sin restriccion de ventana) ===")
ds_clima = construir_dataset_diario(incluir_clima=True)
res_clima, modelo_clima, feats_clima = evaluar_split_oficial(ds_clima, "con clima")

print()
print("=== Comparacion ===")
for split in ["validation", "test"]:
    mae_b, rmse_b = res_base[split]
    mae_c, rmse_c = res_clima[split]
    print(f"{split:<12} mejora MAE: {100*(mae_b-mae_c)/mae_b:+.1f}%   mejora RMSE: {100*(rmse_b-rmse_c)/rmse_b:+.1f}%")

In [ ]:
importancias = pd.Series(modelo_clima.feature_importances_, index=feats_clima).sort_values(ascending=False)
cols_clima = [c for c in feats_clima if any(c.startswith(v) for v in
              ["t2m_mean","wind10_mean","wind100_mean","ssrd_mean","tcc_mean","tp_mean"])]
peso_clima = importancias.loc[cols_clima].sum()

print("Top 15 features (modelo con clima, serie completa):")
print(importancias.head(15))
print(f"
peso conjunto de las {len(cols_clima)} features climaticas: {peso_clima*100:.1f}%")

**Resultado (ejecutado 18-ago-2026, train 2020-2024 / val 2025 / test 2026)**:

| Modelo | MAE val | RMSE val | MAE test | RMSE test |
|---|---|---|---|---|
| Sin clima | 21.10 €/MWh | 26.43 €/MWh | 27.93 €/MWh | 34.52 €/MWh |
| Con clima | 21.30 €/MWh | 26.47 €/MWh | 27.73 €/MWh | 34.11 €/MWh |
| **Mejora** | **-1.0%** | **-0.1%** | **+0.7%** | **+1.2%** |

Las 18 features climáticas caen a **2.9%** de la importancia total (antes: 31-35%).

**Esto revisa la conclusión de la sección 10** — con la serie completa y el split oficial, el
clima prácticamente deja de aportar (ganancia marginal en test, incluso ligeramente peor en
validation). Dos motivos que explican el cambio, visibles en las propias importancias:

1. **`gas_mibgas` domina** (38% de la importancia) — el set de train (2020-2024) incluye la
   crisis energética 2021-2022, donde el precio del gas explica la varianza del precio muchísimo
   más que cualquier variable meteorológica. En la ventana de 19 meses de la sección 10 (todo
   2025-2026, ya sin crisis), el gas pesaba mucho menos y el clima podía "asomar".
2. **Los lags de precio ya existen** (`precio_real_*_lag1d` es la 2ª feature más importante,
   33%) — no existían cuando se corrió el experimento de la sección 10. Es plausible que antes
   el clima estuviera capturando indirectamente parte de la señal de "condiciones recientes del
   mercado" que ahora capturan los lags de forma mucho más directa y eficiente.

**No invalida la decisión del backfill** — era la decisión correcta con la información
disponible entonces, y confirmar esto con datos también es un resultado útil: dice que el
esfuerzo de integrar clima como feature tabular no es prioritario ahora que hay lags de precio;
el valor del clima probablemente esté más en la capa espacio-temporal (CNN+LSTM/Transformer con
los tensores ERA5 crudos) que como agregado diario en un modelo tabular como este.